
Input: `outputs/step3_models.joblib`.
Output: `outputs/train_vs_validation.csv`, figure `slide_figures/trainval.png`.

In [ ]:
# Working folder: holds HABs_ScrippsPier_v3.xlsx and the outputs/ folder shared by all steps.
# In Colab the folder is on Google Drive, so every step notebook can read what the previous step saved.
import os
WORK_FOLDER = "/content/drive/MyDrive/BIF633_Progress2"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(WORK_FOLDER, exist_ok=True)
    os.chdir(WORK_FOLDER)
except ImportError:
    pass  # not running in Colab: use the current folder
os.makedirs("outputs", exist_ok=True)
print("Working folder:", os.getcwd())

In [ ]:
# Load everything saved by the previous step
import joblib
globals().update(joblib.load("outputs/step3_models.joblib"))
print("Loaded outputs/step3_models.joblib")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance

# The models predict log(1 + Chl-a). Convert back to mg/m3 before scoring.
def to_mg(log_values):
    return np.clip(np.expm1(np.asarray(log_values)), 0, None)  # same as the team's convert_log_prediction_to_original

def rmse(observed, predicted):
    return np.sqrt(mean_squared_error(observed, predicted))

def mae(observed, predicted):
    return mean_absolute_error(observed, predicted)

# Observed Chl-a (mg/m3) for the training and validation weeks
observed_train = to_mg(y_train)
observed_validation = to_mg(y_validation)

# The three core models and the feature set used for "set C" by each model
core_models = ["Linear Regression", "Random Forest", "SVR"]
set_C_name = {
    "Linear Regression": "C_Linear_Selected",   # 13 features kept after the VIF check
    "Random Forest": "C_Full_Temporal",
    "SVR": "C_Full_Temporal",
}
short_name = {
    "A_Environmental": "A",
    "B_Environmental_Seasonal": "B",
    "C_Full_Temporal": "C",
    "C_Linear_Selected": "C",
}
print("Ready")

In [ ]:
rows = []
for model_name in core_models:
    for feature_set in ["A_Environmental", "B_Environmental_Seasonal", set_C_name[model_name]]:
        model = trained_models[f"{model_name}__{feature_set}"]["best_estimator"]
        X_train_set = processed_feature_sets[feature_set]["X_train"]
        X_validation_set = processed_feature_sets[feature_set]["X_validation"]

        # Errors on the log scale, so the 467 mg/m3 bloom in 2020 does not dominate
        rows.append({
            "Model": model_name,
            "Feature set": short_name[feature_set],
            "Train error": rmse(y_train, model.predict(X_train_set)),
            "Validation error": rmse(y_validation, model.predict(X_validation_set)),
        })

train_vs_validation = pd.DataFrame(rows)
train_vs_validation["Gap"] = train_vs_validation["Validation error"] - train_vs_validation["Train error"]
display(train_vs_validation.round(3))

# Plot: ring = training error, dot = validation error
labels = train_vs_validation["Model"] + " · " + train_vs_validation["Feature set"]
y_positions = range(len(train_vs_validation))
fig, ax = plt.subplots(figsize=(9, 5))
ax.hlines(y_positions, train_vs_validation["Train error"], train_vs_validation["Validation error"], color="lightgrey")
ax.scatter(train_vs_validation["Train error"], y_positions, s=120, facecolors="none", edgecolors="olive", label="Training")
ax.scatter(train_vs_validation["Validation error"], y_positions, s=60, color="purple", label="Validation")
ax.set_yticks(list(y_positions))
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel("Error on log(1 + Chl-a), lower is better")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
train_vs_validation.to_csv("outputs/train_vs_validation.csv", index=False)
print("Saved outputs/train_vs_validation.csv")

In [ ]:
import os
import matplotlib.dates as mdates
os.makedirs("slide_figures", exist_ok=True)  # figures go here
plt.rcParams.update({"font.size": 18, "axes.spines.top": False, "axes.spines.right": False})
PLUM, TEAL, OLIVE, GREY = "#8A3560", "#00958F", "#A4983A", "#B3B0AA"


# Slide 6: training vs validation error (log scale)
tv = train_vs_validation.iloc[::-1].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(11.4, 6.7))
for i, row in tv.iterrows():
    ax.plot([row["Train error"], row["Validation error"]], [i, i], color=GREY, linewidth=2.5)
ax.scatter(tv["Train error"], tv.index, s=260, facecolors="none", edgecolors=OLIVE, linewidth=3, label="Training")
ax.scatter(tv["Validation error"], tv.index, s=120, color=PLUM, label="Validation")
ax.set_yticks(tv.index)
ax.set_yticklabels(tv["Model"] + " · " + tv["Feature set"])
for line_y in (2.5, 5.5):
    ax.axhline(line_y, color="#E4E1DC", linewidth=1)
ax.tick_params(axis="y", length=0)
ax.spines["left"].set_visible(False)
ax.set_xlim(0.15, 0.68)
ax.set_xlabel("Error on log(1 + Chl-a), lower is better")
ax.legend(frameon=False, loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2)
plt.tight_layout()
plt.savefig("slide_figures/trainval.png", dpi=200)
plt.show()